# Practical 10: Feature Scaling (StandardScaler vs MinMaxScaler) & Train-Test Split

**UNIT 3: Preparing Big Data for Machine Learning**

**Problem Statement:** Write a Python program to perform feature scaling and dataset partitioning. Apply StandardScaler (Z-score normalization) and MinMaxScaler to numerical features, compare their transformed distributions, and split the dataset into training (80%) and testing (20%) sets using train_test_split(..., test_size=0.2, random_state=42).

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)
gen = np.random.default_rng(7)
n = 120
age = gen.integers(20, 58, n).astype(float)
salary = np.round(24000 + age * 850 + gen.normal(0, 5500, n), -2)
orders = pd.DataFrame({
    "Country": gen.choice(["France", "Spain", "Germany"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + gen.normal(0, 8000, n) > 52000, "Yes", "No"),
})
orders.loc[gen.choice(n, 6, replace=False), "Age"] = np.nan
orders.loc[gen.choice(n, 6, replace=False), "Salary"] = np.nan
print(orders.head(8))

num_cols = ["Age", "Salary"]
orders[num_cols] = orders[num_cols].fillna(orders[num_cols].median())
print(orders[num_cols].isna().sum())

std_scaled = pd.DataFrame(StandardScaler().fit_transform(orders[num_cols]), columns=num_cols)
mm_scaled = pd.DataFrame(MinMaxScaler().fit_transform(orders[num_cols]), columns=num_cols)

stats = ["mean", "std", "min", "max"]
print("=== Original ===")
print(orders[num_cols].describe().loc[stats].round(3))
print("\n=== StandardScaler (mean ≈ 0, std ≈ 1) ===")
print(std_scaled.describe().loc[stats].round(3))
print("\n=== MinMaxScaler (range [0, 1]) ===")
print(mm_scaled.describe().loc[stats].round(3))

X = pd.get_dummies(orders.drop(columns=["Purchased"]), columns=["Country"], dtype=int)
y = orders["Purchased"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Total rows: {len(X)} | Train: {X_train.shape} | Test: {X_test.shape}")

scaler = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = scaler.transform(X_test[num_cols])

print("Scaled TRAIN stats (mean ≈ 0, std ≈ 1):")
print(X_train_s[num_cols].describe().loc[["mean", "std"]].round(3))
print("\nScaled TEST stats (close to, but not exactly, 0/1 – as expected):")
print(X_test_s[num_cols].describe().loc[["mean", "std"]].round(3))

,Country,Age,Salary,Purchased
0,Germany,55.0,67000.0,Yes
1,Germany,43.0,61700.0,Yes
2,Germany,45.0,59700.0,Yes
3,Spain,54.0,70600.0,Yes
4,Germany,41.0,52300.0,Yes
5,Germany,49.0,62500.0,Yes
6,France,51.0,66300.0,Yes
7,France,28.0,52700.0,Yes


Age       0
Salary    0
dtype: int64
=== Original ===

,Age,Salary
mean,39.858,56680.833
std,10.512,10457.538
min,20.000,34800.000
max,57.000,80500.000



=== StandardScaler (mean ≈ 0, std ≈ 1) ===

,Age,Salary
mean,-0.000,-0.000
std,1.004,1.004
min,-1.897,-2.101
max,1.637,2.287


...

,Age,Salary
mean,0.537,0.479
std,0.284,0.229
min,0.000,0.000
max,1.000,1.000


Total rows: 120 | Train: (90, 5) | Test: (30, 5)
Scaled TRAIN stats (mean ≈ 0, std ≈ 1):

,Age,Salary
mean,-0.000,-0.000
std,1.006,1.006



Scaled TEST stats (close to, but not exactly, 0/1 – as expected):

,Age,Salary
mean,0.327,0.126
std,1.019,1.203
